In [ ]:
import numpy as np
import sys

class HierarchicalClustering:
    def __init__(self, linkage='average'):
        if linkage not in ['single', 'complete', 'average', 'ward']:
            raise ValueError(f"Неизвестный метод связи: {linkage}")
        self.linkage = linkage
        self.linkage_matrix = None

    def euclidean_distance(self, o1, o2):
        """Евклидово расстояние между двумя точками"""
        return np.sqrt(np.sum((np.array(o1) - np.array(o2)) ** 2))

    def _compute_distance(self, cluster1, cluster2, X):
        """Вычисление расстояния между двумя кластерами"""
        # Собираем все попарные расстояния между точками двух кластеров
        points1 = X[cluster1]
        points2 = X[cluster2]
        
        if self.linkage == 'single':
            # Минимальное расстояние между точками кластеров
            min_dist = float('inf')
            for p1 in points1:
                for p2 in points2:
                    dist = self.euclidean_distance(p1, p2)
                    if dist < min_dist:
                        min_dist = dist
            return min_dist

        elif self.linkage == 'complete':
            # Максимальное расстояние между точками кластеров
            max_dist = 0
            for p1 in points1:
                for p2 in points2:
                    dist = self.euclidean_distance(p1, p2)
                    if dist > max_dist:
                        max_dist = dist
            return max_dist

        elif self.linkage == 'average':
            # Среднее расстояние между всеми парами точек
            total_dist = 0
            count = 0
            for p1 in points1:
                for p2 in points2:
                    total_dist += self.euclidean_distance(p1, p2)
                    count += 1
            return total_dist / count if count > 0 else 0

        elif self.linkage == 'ward':
            # Метод Уорда: минимизация увеличения дисперсии
            # Формула: sqrt(2 * n1 * n2 / (n1 + n2)) * ||c1 - c2||
            n1 = len(cluster1)
            n2 = len(cluster2)
            centroid1 = np.mean(points1, axis=0)
            centroid2 = np.mean(points2, axis=0)
            dist = self.euclidean_distance(centroid1, centroid2)
            return np.sqrt(2 * n1 * n2 / (n1 + n2)) * dist

    def fit(self, X):
        X = np.array(X)
        n_samples = X.shape[0]

        # Инициализация: каждый объект — отдельный кластер
        clusters = [[i] for i in range(n_samples)]

        # Матрица связей: (n_samples - 1) строк по 4 столбца
        self.linkage_matrix = np.zeros((n_samples - 1, 4))

        # ID кластеров
        cluster_ids = list(range(n_samples))
        next_cluster_id = n_samples

        # Итеративное объединение кластеров
        for step in range(n_samples - 1):
            min_dist = float('inf')
            merge_i, merge_j = 0, 1

            # Поиск пары кластеров с минимальным расстоянием
            for i in range(len(clusters)):
                for j in range(i + 1, len(clusters)):
                    dist = self._compute_distance(clusters[i], clusters[j], X)
                    if dist < min_dist:
                        min_dist = dist
                        merge_i, merge_j = i, j

            # Объединение кластеров
            new_cluster = clusters[merge_i] + clusters[merge_j]
            new_id = next_cluster_id
            next_cluster_id += 1

            # Сохранение информации в матрицу связей
            self.linkage_matrix[step, 0] = cluster_ids[merge_i]
            self.linkage_matrix[step, 1] = cluster_ids[merge_j]
            self.linkage_matrix[step, 2] = min_dist
            self.linkage_matrix[step, 3] = len(new_cluster)

            # Обновление списка кластеров и их ID
            # Удаляем в обратном порядке, чтобы не нарушить индексацию
            del clusters[max(merge_i, merge_j)]
            del clusters[min(merge_i, merge_j)]
            del cluster_ids[max(merge_i, merge_j)]
            del cluster_ids[min(merge_i, merge_j)]
            
            # Добавляем новый кластер
            clusters.append(new_cluster)
            cluster_ids.append(new_id)

        return self

    def get_labels_by_height(self, height, n_samples):
        """Разрезает дендрограмму на заданной высоте"""
        # Инициализация: каждый объект — отдельный кластер
        parent = list(range(2 * n_samples - 1))
        children = {i: [i] for i in range(2 * n_samples - 1)}
        
        # Проходим по шагам объединения
        for step in range(n_samples - 1):
            if self.linkage_matrix[step, 2] <= height:
                c1 = int(self.linkage_matrix[step, 0])
                c2 = int(self.linkage_matrix[step, 1])
                new_id = n_samples + step
                
                # Объединяем кластеры
                children[new_id] = children[c1] + children[c2]
                
                # Обновляем родителей
                for child in children[c1] + children[c2]:
                    parent[child] = new_id
        
        # Находим корневые кластеры и назначаем метки
        roots = []
        for i in range(n_samples):
            root = i
            while parent[root] != root:
                root = parent[root]
            if root not in roots:
                roots.append(root)
        
        # Создаём словарь: root -> метка кластера
        root_to_label = {root: idx for idx, root in enumerate(sorted(roots))}
        
        # Назначаем метки точкам
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            root = i
            while parent[root] != root:
                root = parent[root]
            labels[i] = root_to_label[root]
        
        return labels

    def predict(self, n_clusters=None, height=None):
        """Предсказание меток кластеров"""
        if self.linkage_matrix is None:
            raise ValueError("Сначала вызовите метод fit()")
            
        n_samples = self.linkage_matrix.shape[0] + 1

        if height is not None:
            return self.get_labels_by_height(height, n_samples)

        # Граничные случаи для n_clusters
        if n_clusters is None:
            n_clusters = 1
        if n_clusters <= 1:
            return np.zeros(n_samples, dtype=int)
        if n_clusters >= n_samples:
            return np.arange(n_samples, dtype=int)

        # Находим высоту среза для заданного числа кластеров
        # Если нужно k кластеров, делаем (n_samples - k) объединений
        n_merges_needed = n_samples - n_clusters
        
        if n_merges_needed <= 0:
            cutoff_distance = 0
        elif n_merges_needed >= n_samples - 1:
            cutoff_distance = self.linkage_matrix[-1, 2] + 1
        else:
            # Берём расстояние после нужного числа объединений
            cutoff_distance = self.linkage_matrix[n_merges_needed - 1, 2]
            # Добавляем небольшой отступ, чтобы отрезать ровно нужное число кластеров
            if n_merges_needed < n_samples - 1:
                cutoff_distance = (cutoff_distance + self.linkage_matrix[n_merges_needed, 2]) / 2

        return self.get_labels_by_height(cutoff_distance, n_samples)


# Считывание данных и запуск
def main():
    data = sys.stdin.read().strip().split()
    if not data:
        return
    
    k = int(data[0])
    linkage = data[1]
    
    points = []
    for i in range(2, len(data), 2):
        x = float(data[i])
        y = float(data[i + 1])
        points.append([x, y])
    
    X = np.array(points)
    
    # Обучение и предсказание
    hc = HierarchicalClustering(linkage=linkage)
    hc.fit(X)
    labels = hc.predict(n_clusters=k)
    
    # Вывод результатов
    for label in labels:
        print(label)

if __name__ == "__main__":
    main()